<div style="background:#F8FAFC; border:1px solid #E2E8F0; border-left:5px solid #0D9488; border-radius:8px; padding:28px 32px; font-family:'Segoe UI', Helvetica, Arial, sans-serif;">

<div style="color:#0F766E; font-size:11px; font-weight:700; letter-spacing:1.8px; text-transform:uppercase; margin-bottom:10px;">
Computational Chemistry &nbsp;·&nbsp; Research Project
</div>

<div style="color:#0F2742; font-size:28px; font-weight:700; line-height:1.3; margin-bottom:8px;">
AqSolDB: Molecular Aqueous Solubility Prediction
</div>

<div style="color:#334155; font-size:15px; line-height:1.7; margin-bottom:18px;">
A reproducible cheminformatics and machine learning framework for molecular property prediction with scaffold-based validation.
</div>

<div style="border-top:1px solid #CBD5E1; padding-top:12px; color:#475569; font-size:12px; line-height:1.7;">
<strong style="color:#0F766E;">Domain:</strong> Computational Chemistry · Cheminformatics · Molecular Machine Learning
</div>

</div>


<div style="padding: 12px 18px; margin: 15px 0 12px 0; color: #0F2742; font-size: 18px; font-weight: bold; background-color: #F0FDFA; border-left: 6px solid #0D9488; border-radius: 4px;">
    01. Project Overview
</div>

### Problem Statement
Aqueous solubility ($\log S$) is a foundational physicochemical property in early-stage drug discovery and chemical development. It dictates compound developability, oral bioavailability, formulation viability, and overall ADMET (Absorption, Distribution, Metabolism, Excretion, and Toxicity) profiles. Experimental determination via shake-flask or potentiometric assays is low-throughput and resource-intensive, necessitating robust, predictive Quantitative Structure-Property Relationship (QSPR) models.

### Objectives
- Build an end-to-end, reproducible regression pipeline predicting aqueous solubility directly from canonical SMILES strings.
- Enforce strict chemoinformatics validation protocols using Bemis-Murcko scaffold splitting to assess true out-of-scaffold generalization.
- Benchmark a comprehensive suite of candidate regressors and optimize hyperparameters using scaffold-aware cross-validation.
- Deliver a serialized, reproducible inference artifact and prediction interface for novel chemical entities.

### Target Definition
- **Target Variable:** `measured log solubility in mols per litre`
- **Unit / Notation:** $\log_{10}(S)$, where $S$ is aqueous solubility in $\text{mol/L}$.

### Dataset Source
- **Dataset:** AqSolDB / Curated Delaney (ESOL) Aqueous Solubility Benchmark.
- **Composition:** 1,128 diverse organic compounds annotated with experimental aqueous solubility and curated SMILES.

### Expected Outcomes
- Standardized molecular curation pipeline with audit statistics.
- Validated Extended-Connectivity Fingerprints (ECFP4 / Morgan fingerprints, radius 2, 1024 bits).
- Scaffold-isolated train, validation, and test partitions with zero scaffold leakage.
- Benchmark leaderboard comparing validation MAE, RMSE, and $R^2$ across linear, kernel, and tree ensemble models.
- Tuned champion regressor evaluated on the untouched test set and exported for production inference.

<div style="padding: 12px 18px; margin: 20px 0 12px 0; color: #0F2742; font-size: 18px; font-weight: bold; background-color: #F0FDFA; border-left: 6px solid #0D9488; border-radius: 4px;">
    02. Environment and Configuration
</div>

Establish environment dependencies, configure deterministic random seeds, import core chemoinformatics and machine learning libraries, and define experiment-wide hyperparameters.

In [ ]:
# Install specialized chemoinformatics and machine learning libraries if running in fresh environment
%pip install -q rdkit deepchem xgboost lightgbm catboost scikit-learn seaborn scipy

In [ ]:
import os
import sys
import random
import time
import copy
import warnings
from pathlib import Path

# Scientific computing & data processing
import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

# Chemoinformatics (RDKit)
import rdkit
from rdkit import Chem
from rdkit.Chem import Descriptors, Draw, rdFingerprintGenerator
from rdkit.Chem.Scaffolds import MurckoScaffold

# Chemoinformatics (DeepChem)
import deepchem as dc

# Machine Learning & Metrics (Scikit-Learn)
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge, BayesianRidge
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    GradientBoostingRegressor,
    HistGradientBoostingRegressor,
)
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import RandomizedSearchCV

# Gradient Boosting frameworks
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

# Plotting & warnings settings
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120
plt.rcParams["font.sans-serif"] = "DejaVu Sans"
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# Global Configuration Parameters
RANDOM_SEED = 42
TARGET_COL = "measured log solubility in mols per litre"
SMILES_COL = "smiles"

# Morgan Fingerprint parameters (ECFP4 equivalent)
FP_RADIUS = 2
FP_NBITS = 1024

# Scaffold Split Fractions
FRAC_TRAIN = 0.8
FRAC_VALID = 0.1
FRAC_TEST = 0.1

# Artifact storage directory
OUTPUT_DIR = Path("saved_models")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Deterministic Seed Configuration
def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)

set_seed(RANDOM_SEED)

print(f"[Configuration Loaded]")
print(f" • Random Seed: {RANDOM_SEED}")
print(f" • RDKit Version: {rdkit.__version__}")
print(f" • DeepChem Version: {dc.__version__}")
print(f" • Fingerprint: Morgan ECFP4 (Radius={FP_RADIUS}, Bits={FP_NBITS})")
print(f" • Split Ratios: {FRAC_TRAIN:.1f} Train / {FRAC_VALID:.1f} Valid / {FRAC_TEST:.1f} Test")

<div style="padding: 12px 18px; margin: 20px 0 12px 0; color: #0F2742; font-size: 18px; font-weight: bold; background-color: #F0FDFA; border-left: 6px solid #0D9488; border-radius: 4px;">
    03. Dataset Loading
</div>

Load the aqueous solubility dataset, resolve runtime environments (Kaggle or local), verify schema definitions, inspect dimensions, and confirm target columns.

In [ ]:
# Define candidate locations for Kaggle and local environments
candidate_paths = [
    Path("/kaggle/input/datasets/mdraselsarker/aqueous-solubility-dataset/aqueous solubility dataset.csv"),
    Path("/kaggle/input/aqueous-solubility-dataset/aqueous solubility dataset.csv"),
    Path("aqueous solubility dataset.csv"),
    Path("../data/aqueous solubility dataset.csv"),
    Path("backend/aqueous solubility dataset.csv"),
]

data_path = None
for p in candidate_paths:
    if p.exists():
        data_path = p
        break

if data_path is None:
    data_path = candidate_paths[0]
    print(f"[Notice] Local file not found on disk. Defaulting to Kaggle path: {data_path}")
else:
    print(f"[Dataset Located] Path: {data_path.resolve()}")

# Load raw dataset
raw_df = pd.read_csv(data_path)

# Verify schema and dimensions
print(f"Dataset Shape: {raw_df.shape[0]} rows × {raw_df.shape[1]} columns\n")
print("Dataset Column Inventory:")
print(raw_df.columns.tolist())

print("\nSchema Summary:")
raw_df.info()

print("\nFirst 5 Records:")
display(raw_df.head(5))

# Confirm presence of mandatory columns
assert TARGET_COL in raw_df.columns, f"Target column '{TARGET_COL}' is missing."
assert SMILES_COL in raw_df.columns, f"SMILES column '{SMILES_COL}' is missing."
print(f"\n[Validation Passed] Identified Target: '{TARGET_COL}' | SMILES: '{SMILES_COL}'")

<div style="padding: 12px 18px; margin: 20px 0 12px 0; color: #0F2742; font-size: 18px; font-weight: bold; background-color: #F0FDFA; border-left: 6px solid #0D9488; border-radius: 4px;">
    3.1 Dataset Inspection
</div>

In [ ]:
import pandas as pd
import numpy as np

# Dataset dimensions
print("Dataset shape:", data.shape)
print("Number of rows:", data.shape[0])
print("Number of columns:", data.shape[1])

# Column names
print("\nColumn names:")
print(data.columns.tolist())

# Data types and memory usage
print("\nDataset information:")
data.info()

# Missing values
print("\nMissing values per column:")
print(data.isnull().sum())

print("\nMissing value percentage:")
print((data.isnull().mean() * 100).round(2))

# Sample records
display(data.head())
display(data.sample(min(5, len(data)), random_state=42))

# Descriptive statistics
display(data.describe(include="all").T)

# Duplicate rows
print("\nExact duplicate rows:", data.duplicated().sum())

# Unique values per column
print("\nUnique values:")
print(data.nunique(dropna=False).sort_values())

# Infinite values in numerical columns
numeric_data = data.select_dtypes(include=np.number)
print("\nInfinite values per numerical column:")
print(np.isinf(numeric_data).sum())

<div style="padding: 12px 18px; margin: 20px 0 12px 0; color: #0F2742; font-size: 18px; font-weight: bold; background-color: #F0FDFA; border-left: 6px solid #0D9488; border-radius: 4px;">
    04. Data Quality and Cleaning
</div>

Perform chemoinformatics data validation: detect missing values, validate and parse molecular SMILES using RDKit, generate canonical isomeric representations, resolve duplicate chemical entities, and produce an audit summary.

In [ ]:
# 1. Missing Value Assessment
missing_counts = raw_df.isnull().sum()
print("Missing Values by Column:")
print(missing_counts[missing_counts > 0] if missing_counts.sum() > 0 else "None found across dataset.")

# 2. SMILES Parsing and Canonicalization
def standardize_smiles(smiles: str):
    """
    Validates a SMILES string with RDKit and converts it into a canonical,
    isomeric representation. Returns None if parsing fails.
    """
    if not isinstance(smiles, str) or not smiles.strip():
        return None
    try:
        mol = Chem.MolFromSmiles(smiles.strip())
        if mol is None:
            return None
        return Chem.MolToSmiles(mol, canonical=True, isomericSmiles=True)
    except Exception:
        return None

df_audit = raw_df.copy()
df_audit["canonical_smiles"] = df_audit[SMILES_COL].apply(standardize_smiles)

initial_count = len(df_audit)
invalid_smiles_count = int(df_audit["canonical_smiles"].isnull().sum())

# Retain only valid chemical structures with target measurements
df_valid = df_audit.dropna(subset=["canonical_smiles", TARGET_COL]).copy()
valid_count = len(df_valid)

# 3. Duplicate Structure Identification and Aggregation
duplicate_count = int(df_valid.duplicated(subset=["canonical_smiles"]).sum())
print(f"\nDuplicate Canonical SMILES Detected: {duplicate_count}")

# In chemoinformatics benchmarks, duplicate measurements should be aggregated (mean)
clean_df = (
    df_valid.groupby("canonical_smiles", as_index=False)
    .agg({
        TARGET_COL: "mean",
        "Compound ID": "first" if "Compound ID" in df_valid.columns else "first"
    })
)
final_count = len(clean_df)

# 4. Data Quality Audit Summary Table
audit_summary = pd.DataFrame({
    "Audit Check": [
        "Initial Records Loaded",
        "Missing Target Records",
        "Invalid / Unparseable SMILES Removed",
        "Valid Molecular Structures",
        "Duplicate Structures Aggregated",
        "Final Curated Clean Dataset"
    ],
    "Count": [
        initial_count,
        int(raw_df[TARGET_COL].isnull().sum()),
        invalid_smiles_count,
        valid_count,
        duplicate_count,
        final_count
    ]
})

print("\n--- Chemoinformatics Quality Audit Summary ---")
display(audit_summary)

<div style="padding: 12px 18px; margin: 20px 0 12px 0; color: #0F2742; font-size: 18px; font-weight: bold; background-color: #F0FDFA; border-left: 6px solid #0D9488; border-radius: 4px;">
    05. Exploratory Data Analysis
</div>

Analyze the target distribution, compute descriptive statistics, extract fundamental physicochemical properties (Lipinski descriptors), examine property-solubility correlations, and visualize representative chemical structures.

In [ ]:
target_vals = clean_df[TARGET_COL].to_numpy()

# 1. Descriptive Statistics of Target
stats_data = {
    "Sample Count": len(target_vals),
    "Mean (mol/L)": np.mean(target_vals),
    "Std Deviation": np.std(target_vals),
    "Median (mol/L)": np.median(target_vals),
    "IQR": stats.iqr(target_vals),
    "Min (mol/L)": np.min(target_vals),
    "Max (mol/L)": np.max(target_vals),
    "Skewness": stats.skew(target_vals),
    "Kurtosis": stats.kurtosis(target_vals),
}

stats_df = pd.DataFrame(list(stats_data.items()), columns=["Metric", "Value"])
print("Descriptive Statistics for Aqueous Solubility:")
display(stats_df.round(4))

# 2. Target Distribution Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(target_vals, kde=True, bins=35, color="#0D9488", edgecolor="white", ax=axes[0])
axes[0].axvline(np.mean(target_vals), color="#E11D48", linestyle="--", linewidth=1.5, label=f"Mean: {np.mean(target_vals):.2f}")
axes[0].axvline(np.median(target_vals), color="#2563EB", linestyle=":", linewidth=1.5, label=f"Median: {np.median(target_vals):.2f}")
axes[0].set_title("Target Distribution (Measured Log Solubility)", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Measured Log Solubility (mol/L)")
axes[0].set_ylabel("Frequency")
axes[0].legend()

sns.boxplot(x=target_vals, color="#99F6E4", ax=axes[1])
axes[1].set_title("Solubility Distribution & Outlier Boxplot", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Measured Log Solubility (mol/L)")

plt.tight_layout()
plt.show()

In [ ]:
# 3. Physicochemical Property Extraction & Correlation
def calculate_descriptors(smiles_series):
    mw, logp, hbd, hba, tpsa, rotb = [], [], [], [], [], []
    for smi in smiles_series:
        mol = Chem.MolFromSmiles(smi)
        mw.append(Descriptors.MolWt(mol))
        logp.append(Descriptors.MolLogP(mol))
        hbd.append(Descriptors.NumHDonors(mol))
        hba.append(Descriptors.NumHAcceptors(mol))
        tpsa.append(Descriptors.TPSA(mol))
        rotb.append(Descriptors.NumRotatableBonds(mol))
    return pd.DataFrame({
        "Molecular Weight": mw,
        "MolLogP (Lipophilicity)": logp,
        "H-Bond Donors": hbd,
        "H-Bond Acceptors": hba,
        "TPSA (Å²)": tpsa,
        "Rotatable Bonds": rotb,
    })

desc_df = calculate_descriptors(clean_df["canonical_smiles"])
desc_df[TARGET_COL] = clean_df[TARGET_COL].values

# Correlation Heatmap
plt.figure(figsize=(9, 6))
sns.heatmap(
    desc_df.corr(),
    annot=True,
    fmt=".2f",
    cmap="Spectral_r",
    center=0,
    linewidths=0.5
)
plt.title("Correlation Matrix: Physicochemical Descriptors vs. Solubility", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

# 4. Molecular Visualizations Across the Solubility Spectrum
sorted_clean_df = clean_df.sort_values(by=TARGET_COL).reset_index(drop=True)

# Select representatives from bottom (insoluble), middle (moderately soluble), and top (highly soluble)
low_mols = [Chem.MolFromSmiles(s) for s in sorted_clean_df.iloc[:3]["canonical_smiles"]]
mid_idx = len(sorted_clean_df) // 2
mid_mols = [Chem.MolFromSmiles(s) for s in sorted_clean_df.iloc[mid_idx-1:mid_idx+2]["canonical_smiles"]]
high_mols = [Chem.MolFromSmiles(s) for s in sorted_clean_df.iloc[-3:]["canonical_smiles"]]

all_mols = low_mols + mid_mols + high_mols
rep_subset = pd.concat([
    sorted_clean_df.iloc[:3],
    sorted_clean_df.iloc[mid_idx-1:mid_idx+2],
    sorted_clean_df.iloc[-3:]
])
mol_labels = [f"LogS: {val:.2f}" for val in rep_subset[TARGET_COL]]

print("\nSelected Representative Structures Across Solubility Spectrum:")
print(" • Row 1: Low Solubility (LogS < -7.0) | Row 2: Moderate (LogS ~ -3.0) | Row 3: High Solubility (LogS > 0.5)")

Draw.MolsToGridImage(
    all_mols,
    molsPerRow=3,
    subImgSize=(280, 200),
    legends=mol_labels
)

<div style="padding: 12px 18px; margin: 20px 0 12px 0; color: #0F2742; font-size: 18px; font-weight: bold; background-color: #F0FDFA; border-left: 6px solid #0D9488; border-radius: 4px;">
    06. Molecular Featurization
</div>

Featurize molecules using standardized Extended-Connectivity Fingerprints (ECFP4 / Morgan Fingerprints, radius 2, 1024 bits) and perform feature matrix validation checks.

In [ ]:
# Initialize modern RDKit Morgan Fingerprint Generator
morgan_gen = rdFingerprintGenerator.GetMorganGenerator(
    radius=FP_RADIUS,
    fpSize=FP_NBITS
)

def mol_to_fp_vector(smiles: str, generator, n_bits: int) -> np.ndarray:
    """Generates a fixed-length numpy Morgan fingerprint bit-vector from SMILES."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return np.zeros(n_bits, dtype=np.float32)
    return generator.GetFingerprintAsNumPy(mol).astype(np.float32)

# Build feature matrix X and target vector y
X = np.vstack([
    mol_to_fp_vector(smi, morgan_gen, FP_NBITS)
    for smi in clean_df["canonical_smiles"]
])
y = clean_df[TARGET_COL].to_numpy(dtype=np.float32)
clean_smiles_list = clean_df["canonical_smiles"].tolist()

# Feature Matrix Validation & Audit
n_samples, n_features = X.shape
nan_count = int(np.isnan(X).sum())
inf_count = int(np.isinf(X).sum())
active_density = (X.sum() / (n_samples * n_features)) * 100
zero_var_bits = int((np.var(X, axis=0) == 0).sum())

fp_audit_df = pd.DataFrame({
    "Featurization Metric": [
        "Feature Matrix Dimensions (X)",
        "Target Vector Dimensions (y)",
        "Missing / NaN Bit Values",
        "Infinite Values",
        "Average Active Bit Density",
        "Zero-Variance (Invariant) Features",
        "Active Informative Features"
    ],
    "Value": [
        f"{n_samples} samples × {n_features} features",
        f"{len(y)} samples",
        f"{nan_count}",
        f"{inf_count}",
        f"{active_density:.2f}%",
        f"{zero_var_bits} / {n_features}",
        f"{n_features - zero_var_bits} / {n_features}"
    ]
})

print("--- Molecular Featurization Audit ---")
display(fp_audit_df)
assert nan_count == 0 and inf_count == 0, "Feature matrix contains NaN or Inf values."
print(" ✓ Feature matrix verified: 1024-bit Morgan vectors constructed successfully.")

<div style="padding: 12px 18px; margin: 20px 0 12px 0; color: #0F2742; font-size: 18px; font-weight: bold; background-color: #F0FDFA; border-left: 6px solid #0D9488; border-radius: 4px;">
    07. Scaffold-Based Splitting
</div>

Partition dataset into training (80%), validation (10%), and test (10%) sets using Bemis-Murcko scaffold splitting. This enforces strict chemical separation between partitions and tests out-of-scaffold generalization.

In [ ]:
# Create DeepChem NumpyDataset with canonical SMILES preserved as identifiers
dc_dataset = dc.data.NumpyDataset(
    X=X,
    y=y.reshape(-1, 1),
    ids=np.array(clean_smiles_list)
)

# Apply DeepChem Bemis-Murcko ScaffoldSplitter
scaffold_splitter = dc.splits.ScaffoldSplitter()

train_dataset, valid_dataset, test_dataset = scaffold_splitter.train_valid_test_split(
    dc_dataset,
    frac_train=FRAC_TRAIN,
    frac_valid=FRAC_VALID,
    frac_test=FRAC_TEST,
    seed=RANDOM_SEED
)

# Extract numpy arrays for model training and evaluation
X_train, y_train = train_dataset.X, train_dataset.y.ravel()
X_valid, y_valid = valid_dataset.X, valid_dataset.y.ravel()
X_test, y_test = test_dataset.X, test_dataset.y.ravel()

ids_train = train_dataset.ids
ids_valid = valid_dataset.ids
ids_test = test_dataset.ids

# Partition Documentation Table
split_summary_df = pd.DataFrame({
    "Partition": ["Training Set", "Validation Set", "Test Set (Untouched)", "Total Clean Dataset"],
    "Samples (N)": [len(train_dataset), len(valid_dataset), len(test_dataset), len(dc_dataset)],
    "Proportion (%)": [
        f"{(len(train_dataset)/len(dc_dataset))*100:.1f}%",
        f"{(len(valid_dataset)/len(dc_dataset))*100:.1f}%",
        f"{(len(test_dataset)/len(dc_dataset))*100:.1f}%",
        "100.0%"
    ],
    "Mean Solubility": [f"{y_train.mean():.3f}", f"{y_valid.mean():.3f}", f"{y_test.mean():.3f}", f"{y.mean():.3f}"],
    "Std Solubility": [f"{y_train.std():.3f}", f"{y_valid.std():.3f}", f"{y_test.std():.3f}", f"{y.std():.3f}"]
})

print("--- Scaffold Partition Documentation ---")
display(split_summary_df)

# Verify Zero Scaffold Leakage Across Partitions
def extract_unique_scaffolds(smiles_array):
    scaffolds = set()
    for smi in smiles_array:
        mol = Chem.MolFromSmiles(smi)
        if mol:
            scaffold = MurckoScaffold.MurckoScaffoldSmiles(mol=mol, includeChirality=False)
            scaffolds.add(scaffold)
    return scaffolds

train_scaffolds = extract_unique_scaffolds(ids_train)
valid_scaffolds = extract_unique_scaffolds(ids_valid)
test_scaffolds = extract_unique_scaffolds(ids_test)

leak_train_valid = len(train_scaffolds.intersection(valid_scaffolds))
leak_train_test = len(train_scaffolds.intersection(test_scaffolds))
leak_valid_test = len(valid_scaffolds.intersection(test_scaffolds))

print("\n--- Scaffold Leakage Audit ---")
print(f" • Unique Scaffolds: Train={len(train_scaffolds)}, Valid={len(valid_scaffolds)}, Test={len(test_scaffolds)}")
print(f" • Train ∩ Valid Scaffold Overlap: {leak_train_valid} (Must be 0)")
print(f" • Train ∩ Test Scaffold Overlap:  {leak_train_test} (Must be 0)")
print(f" • Valid ∩ Test Scaffold Overlap:  {leak_valid_test} (Must be 0)")

assert leak_train_valid == 0 and leak_train_test == 0 and leak_valid_test == 0, "Scaffold leakage detected!"
print(" ✓ Scaffold isolation verified: 0% scaffold leakage between all partitions.")

<div style="padding: 12px 18px; margin: 20px 0 12px 0; color: #0F2742; font-size: 18px; font-weight: bold; background-color: #F0FDFA; border-left: 6px solid #0D9488; border-radius: 4px;">
    08. Baseline Evaluation
</div>

Train a naive mean-prediction dummy regressor on the training partition and evaluate baseline MAE, RMSE, and $R^2$ on the validation partition (keeping the test set untouched).

In [ ]:
# Fit Mean Dummy Regressor on training set
baseline_model = DummyRegressor(strategy="mean")
baseline_model.fit(X_train, y_train)

# Evaluate strictly on validation set
y_valid_baseline = baseline_model.predict(X_valid)

baseline_mae = mean_absolute_error(y_valid, y_valid_baseline)
baseline_rmse = np.sqrt(mean_squared_error(y_valid, y_valid_baseline))
baseline_r2 = r2_score(y_valid, y_valid_baseline)

baseline_results_df = pd.DataFrame([{
    "Model": "Mean-Prediction Baseline",
    "Validation MAE": baseline_mae,
    "Validation RMSE": baseline_rmse,
    "Validation R²": baseline_r2,
    "Prediction Value": y_train.mean()
}])

print("--- Baseline Model Performance (Validation Set) ---")
display(baseline_results_df.round(4))

# Baseline Validation Prediction Plot
plt.figure(figsize=(7, 5))
plt.scatter(y_valid, y_valid_baseline, alpha=0.6, color="#0D9488", label="Validation Predictions")
plt.axhline(y_train.mean(), color="#E11D48", linestyle="--", linewidth=1.5, label=f"Train Mean Prediction ({y_train.mean():.2f})")
plt.title("Baseline Model: Validation Actual vs. Predicted", fontsize=12, fontweight="bold")
plt.xlabel("Actual Validation Solubility (mol/L)")
plt.ylabel("Baseline Predicted Solubility (mol/L)")
plt.legend()
plt.tight_layout()
plt.show()

<div style="padding: 12px 18px; margin: 20px 0 12px 0; color: #0F2742; font-size: 18px; font-weight: bold; background-color: #F0FDFA; border-left: 6px solid #0D9488; border-radius: 4px;">
    09. Model Benchmarking
</div>

Train a diverse suite of candidate regressors across linear, kernel, tree ensemble, and gradient boosting algorithms on the training partition. Benchmark validation MAE, RMSE, and $R^2$ to identify top architectures.

In [ ]:
# Define candidate regressors covering diverse inductive biases
candidate_regressors = {
    "Ridge Regression": Ridge(alpha=1.0, random_state=RANDOM_SEED),
    "Bayesian Ridge": BayesianRidge(),
    "Support Vector Regressor (SVR)": SVR(kernel="rbf", C=1.0, epsilon=0.1),
    "K-Nearest Neighbors (KNN)": KNeighborsRegressor(n_neighbors=5, weights="distance"),
    "Random Forest": RandomForestRegressor(n_estimators=150, max_depth=15, random_state=RANDOM_SEED, n_jobs=-1),
    "Extra Trees": ExtraTreesRegressor(n_estimators=150, max_depth=15, random_state=RANDOM_SEED, n_jobs=-1),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=100, learning_rate=0.08, max_depth=4, random_state=RANDOM_SEED),
    "HistGradientBoosting": HistGradientBoostingRegressor(max_iter=100, learning_rate=0.08, random_state=RANDOM_SEED),
    "XGBoost": XGBRegressor(n_estimators=150, learning_rate=0.05, max_depth=5, random_state=RANDOM_SEED, n_jobs=-1, verbosity=0)
}

benchmark_records = []
trained_models = {}
val_predictions = {}

print("Training Candidate Regressors...")
for name, model in candidate_regressors.items():
    t0 = time.time()
    try:
        model.fit(X_train, y_train)
        y_pred_val = model.predict(X_valid)
        t_fit = time.time() - t0
        
        mae = mean_absolute_error(y_valid, y_pred_val)
        rmse = np.sqrt(mean_squared_error(y_valid, y_pred_val))
        r2 = r2_score(y_valid, y_pred_val)
        
        benchmark_records.append({
            "Model": name,
            "Validation MAE": mae,
            "Validation RMSE": rmse,
            "Validation R²": r2,
            "Fit Time (s)": round(t_fit, 3)
        })
        trained_models[name] = model
        val_predictions[name] = y_pred_val
        print(f" ✓ {name:<32} | RMSE: {rmse:.4f} | MAE: {mae:.4f} | R²: {r2:.4f} ({t_fit:.2f}s)")
    except Exception as err:
        print(f" ✗ {name:<32} | Error: {str(err)}")

# Create benchmark leaderboard ranked by validation RMSE (primary criterion)
benchmark_df = pd.DataFrame(benchmark_records).sort_values(by="Validation RMSE", ascending=True).reset_index(drop=True)

print("\n--- Validation Benchmark Leaderboard ---")
display(benchmark_df.round(4))

In [ ]:
# Comparative Metric Visualizations across Candidate Regressors
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Validation RMSE
sns.barplot(
    data=benchmark_df,
    x="Validation RMSE",
    y="Model",
    palette="viridis",
    ax=axes[0]
)
axes[0].set_title("Validation RMSE — Lower is Better", fontsize=12, fontweight="bold")
axes[0].set_xlabel("RMSE (mol/L)")

# Validation MAE
sns.barplot(
    data=benchmark_df,
    x="Validation MAE",
    y="Model",
    palette="magma",
    ax=axes[1]
)
axes[1].set_title("Validation MAE — Lower is Better", fontsize=12, fontweight="bold")
axes[1].set_xlabel("MAE (mol/L)")

# Validation R²
sns.barplot(
    data=benchmark_df,
    x="Validation R²",
    y="Model",
    palette="crest",
    ax=axes[2]
)
axes[2].set_title("Validation R² — Higher is Better", fontsize=12, fontweight="bold")
axes[2].set_xlabel("R² Score")

plt.tight_layout()
plt.show()

<div style="padding: 12px 18px; margin: 20px 0 12px 0; color: #0F2742; font-size: 18px; font-weight: bold; background-color: #F0FDFA; border-left: 6px solid #0D9488; border-radius: 4px;">
    10. Hyperparameter Optimization
</div>

Tune the top candidate models using scaffold-aware cross-validation on the training set, optimizing tree depth, learning rate, leaf constraints, and regularization parameters.

In [ ]:
# Build Scaffold-Aware Cross-Validation Generator on Training Partition
def build_scaffold_cv(smiles_array, n_splits=5, seed=42):
    """
    Groups training compounds by Bemis-Murcko scaffolds and creates cross-validation
    folds to ensure zero scaffold overlap between inner train and inner validation folds.
    """
    scaffold_map = {}
    for idx, smi in enumerate(smiles_array):
        mol = Chem.MolFromSmiles(smi)
        scaffold = MurckoScaffold.MurckoScaffoldSmiles(mol=mol, includeChirality=False) if mol else ""
        scaffold_map.setdefault(scaffold, []).append(idx)
    
    rng = np.random.RandomState(seed)
    scaffold_clusters = list(scaffold_map.values())
    rng.shuffle(scaffold_clusters)
    
    folds = [[] for _ in range(n_splits)]
    for cluster in scaffold_clusters:
        min_fold = min(range(n_splits), key=lambda i: len(folds[i]))
        folds[min_fold].extend(cluster)
    
    cv_splits = []
    total_indices = set(range(len(smiles_array)))
    for f in range(n_splits):
        val_idx = np.array(folds[f])
        tr_idx = np.array(list(total_indices - set(val_idx)))
        cv_splits.append((tr_idx, val_idx))
    return cv_splits

scaffold_cv = build_scaffold_cv(ids_train, n_splits=5, seed=RANDOM_SEED)
print(f"[Scaffold CV] Created 5-fold scaffold-isolated cross-validation splits on training partition.")

# Define hyperparameter distributions for top candidate families
tuning_specs = {
    "HistGradientBoosting": {
        "estimator": HistGradientBoostingRegressor(random_state=RANDOM_SEED),
        "param_dist": {
            "learning_rate": stats.uniform(0.02, 0.12),
            "max_iter": stats.randint(80, 250),
            "max_leaf_nodes": stats.randint(15, 63),
            "min_samples_leaf": stats.randint(10, 40),
            "l2_regularization": stats.uniform(0.0, 2.0),
        },
        "n_iter": 20
    },
    "Random Forest": {
        "estimator": RandomForestRegressor(random_state=RANDOM_SEED, n_jobs=-1),
        "param_dist": {
            "n_estimators": stats.randint(100, 300),
            "max_depth": stats.randint(8, 25),
            "min_samples_split": stats.randint(2, 10),
            "min_samples_leaf": stats.randint(1, 6),
            "max_features": ["sqrt", "log2", 0.3, 0.5]
        },
        "n_iter": 15
    }
}

tuned_models = {}
tuning_records = []

for name, spec in tuning_specs.items():
    print(f"\nTuning {name} with Scaffold-Aware Cross-Validation...")
    search = RandomizedSearchCV(
        estimator=spec["estimator"],
        param_distributions=spec["param_dist"],
        n_iter=spec["n_iter"],
        scoring="neg_root_mean_squared_error",
        cv=scaffold_cv,
        random_state=RANDOM_SEED,
        n_jobs=-1,
        refit=True
    )
    search.fit(X_train, y_train)
    best_estimator = search.best_estimator_
    tuned_models[name] = best_estimator
    
    # Evaluate tuned model on the validation set
    y_val_tuned = best_estimator.predict(X_valid)
    tuned_mae = mean_absolute_error(y_valid, y_val_tuned)
    tuned_rmse = np.sqrt(mean_squared_error(y_valid, y_val_tuned))
    tuned_r2 = r2_score(y_valid, y_val_tuned)
    
    # Retrieve default benchmark metrics
    default_row = benchmark_df[benchmark_df["Model"] == name].iloc[0]
    
    tuning_records.append({
        "Model": name,
        "Stage": "Pre-Tuning (Default)",
        "Validation MAE": default_row["Validation MAE"],
        "Validation RMSE": default_row["Validation RMSE"],
        "Validation R²": default_row["Validation R²"],
        "Best Parameters": "Default"
    })
    tuning_records.append({
        "Model": name,
        "Stage": "Post-Tuning (Optimized)",
        "Validation MAE": tuned_mae,
        "Validation RMSE": tuned_rmse,
        "Validation R²": tuned_r2,
        "Best Parameters": str(search.best_params_)
    })
    print(f" ✓ {name} Tuned Valid RMSE: {tuned_rmse:.4f} (Pre-Tuning: {default_row['Validation RMSE']:.4f})")

# Tabulate optimization results
tuning_comparison_df = pd.DataFrame(tuning_records)
print("\n--- Hyperparameter Optimization Comparison ---")
display(tuning_comparison_df[["Model", "Stage", "Validation MAE", "Validation RMSE", "Validation R²"]].round(4))

<div style="padding: 12px 18px; margin: 20px 0 12px 0; color: #0F2742; font-size: 18px; font-weight: bold; background-color: #F0FDFA; border-left: 6px solid #0D9488; border-radius: 4px;">
    11. Final Model Selection
</div>

Select the final candidate regressor based strictly on validation performance metrics. Retrain on the combined training and validation data, serialize model artifacts, and verify integrity.

In [ ]:
# Compile all candidate models and their validation performances
candidate_pool = {}

for name, model in trained_models.items():
    preds = model.predict(X_valid)
    candidate_pool[name] = {
        "model": model,
        "rmse": np.sqrt(mean_squared_error(y_valid, preds)),
        "mae": mean_absolute_error(y_valid, preds),
        "r2": r2_score(y_valid, preds),
    }

for name, model in tuned_models.items():
    preds = model.predict(X_valid)
    candidate_pool[f"Tuned {name}"] = {
        "model": model,
        "rmse": np.sqrt(mean_squared_error(y_valid, preds)),
        "mae": mean_absolute_error(y_valid, preds),
        "r2": r2_score(y_valid, preds),
    }

# Rank strictly by Validation RMSE
ranked_candidates = sorted(candidate_pool.items(), key=lambda x: x[1]["rmse"])
champion_name, champion_stats = ranked_candidates[0]
champion_instance = champion_stats["model"]

print("==================================================")
print(f"Champion Model Selected: {champion_name}")
print(f" • Validation RMSE: {champion_stats['rmse']:.4f}")
print(f" • Validation MAE:  {champion_stats['mae']:.4f}")
print(f" • Validation R²:   {champion_stats['r2']:.4f}")
print("==================================================")

# Refit on combined Train + Validation partition to leverage complete non-test data
X_train_val = np.vstack([X_train, X_valid])
y_train_val = np.concatenate([y_train, y_valid])

final_model = copy.deepcopy(champion_instance)
final_model.fit(X_train_val, y_train_val)

# Persist champion model artifact
model_filepath = OUTPUT_DIR / "solubility_prediction_model.joblib"
joblib.dump(final_model, model_filepath)

# Persist metadata for downstream deployment
metadata_filepath = OUTPUT_DIR / "pipeline_metadata.joblib"
pipeline_metadata = {
    "model_name": champion_name,
    "model_class": str(type(final_model)),
    "fingerprint_type": "Morgan / ECFP4",
    "fingerprint_radius": FP_RADIUS,
    "fingerprint_nbits": FP_NBITS,
    "target_column": TARGET_COL,
    "target_unit": "log10(mol/L)",
    "validation_rmse": float(champion_stats["rmse"]),
    "validation_mae": float(champion_stats["mae"]),
    "validation_r2": float(champion_stats["r2"]),
}
joblib.dump(pipeline_metadata, metadata_filepath)

print(f"\n[Artifact Saved] Model: {model_filepath.resolve()}")
print(f"[Artifact Saved] Metadata: {metadata_filepath.resolve()}")

# Verify artifact reload integrity
reloaded_model = joblib.load(model_filepath)
test_sample_preds = reloaded_model.predict(X_test[:5])
direct_preds = final_model.predict(X_test[:5])
assert np.allclose(test_sample_preds, direct_preds), "Model artifact verification failed!"
print(" ✓ Champion model artifact reloaded and verified successfully.")

<div style="padding: 12px 18px; margin: 20px 0 12px 0; color: #0F2742; font-size: 18px; font-weight: bold; background-color: #F0FDFA; border-left: 6px solid #0D9488; border-radius: 4px;">
    12. Final Test Evaluation
</div>

Evaluate the champion model once on the untouched test partition. Document generalization performance, inspect publication-grade diagnostic plots (parity, residual, and error distribution), and provide a production inference interface.

In [ ]:
# Perform single-pass evaluation on untouched test set
y_test_pred = final_model.predict(X_test)

test_mae = mean_absolute_error(y_test, y_test_pred)
test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
test_r2 = r2_score(y_test, y_test_pred)

# Compute training set metrics for generalization comparison
y_train_pred = final_model.predict(X_train)
train_mae = mean_absolute_error(y_train, y_train_pred)
train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
train_r2 = r2_score(y_train, y_train_pred)

# Generalization Performance Table
generalization_report = pd.DataFrame({
    "Partition": ["Training Set", "Validation Set", "Test Set (Untouched)"],
    "MAE (mol/L)": [train_mae, champion_stats["mae"], test_mae],
    "RMSE (mol/L)": [train_rmse, champion_stats["rmse"], test_rmse],
    "R² Score": [train_r2, champion_stats["r2"], test_r2]
})

print("==================================================")
print("FINAL GENERALIZATION PERFORMANCE REPORT")
print("==================================================")
display(generalization_report.round(4))

# Export test predictions
test_export_df = pd.DataFrame({
    "SMILES": ids_test,
    "Actual_LogS": y_test,
    "Predicted_LogS": y_test_pred,
    "Residual": y_test - y_test_pred,
    "Absolute_Error": np.abs(y_test - y_test_pred)
})
test_pred_path = OUTPUT_DIR / "test_set_predictions.csv"
test_export_df.to_csv(test_pred_path, index=False)
print(f"\n[Test Set Export] Saved test set predictions to: {test_pred_path}")

<div style="padding: 12px 18px; margin: 20px 0 12px 0; color: #0F2742; font-size: 18px; font-weight: bold; background-color: #F0FDFA; border-left: 6px solid #0D9488; border-radius: 4px;">
    12.1 Save Final Model
</div>

In [ ]:
import joblib

model_path = OUTPUT_DIR / "final_solubility_model.joblib"

joblib.dump(final_model, model_path)

print(f"\n[Model Export] Final model saved to: {model_path}")

# Verify that the saved model can be loaded
loaded_model = joblib.load(model_path)

# Confirm predictions remain identical after serialization
loaded_predictions = loaded_model.predict(X_test)

assert np.allclose(
    y_test_pred,
    loaded_predictions,
    rtol=1e-7,
    atol=1e-8
), "Saved model predictions differ from original predictions!"

print("[Verification] Model successfully saved, loaded, and verified.")

In [ ]:
# Publication-Grade Diagnostic Visualizations
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Parity Plot (Actual vs Predicted)
axes[0].scatter(y_test, y_test_pred, alpha=0.7, color="#0D9488", edgecolors="white", linewidth=0.5, label="Test Compounds")
min_val = min(y_test.min(), y_test_pred.min()) - 0.5
max_val = max(y_test.max(), y_test_pred.max()) + 0.5
axes[0].plot([min_val, max_val], [min_val, max_val], color="#E11D48", linestyle="--", linewidth=1.5, label="Ideal (y = x)")
axes[0].plot([min_val, max_val], [min_val + 1, max_val + 1], color="gray", linestyle=":", label="±1.0 Log Unit Error")
axes[0].plot([min_val, max_val], [min_val - 1, max_val - 1], color="gray", linestyle=":")
axes[0].set_xlim([min_val, max_val])
axes[0].set_ylim([min_val, max_val])
axes[0].set_title(f"Parity Plot: Actual vs. Predicted (R² = {test_r2:.3f})", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Experimental Log Solubility (mol/L)")
axes[0].set_ylabel("Predicted Log Solubility (mol/L)")
axes[0].legend(loc="upper left")

# 2. Residual Plot
residuals = y_test - y_test_pred
axes[1].scatter(y_test_pred, residuals, alpha=0.7, color="#2563EB", edgecolors="white", linewidth=0.5)
axes[1].axhline(0, color="#E11D48", linestyle="--", linewidth=1.5)
axes[1].axhline(1, color="gray", linestyle=":")
axes[1].axhline(-1, color="gray", linestyle=":")
axes[1].set_title(f"Residual Plot (RMSE = {test_rmse:.3f})", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Predicted Log Solubility (mol/L)")
axes[1].set_ylabel("Residual (Actual - Predicted)")

# 3. Residual Distribution & Normality Check
sns.histplot(residuals, kde=True, bins=25, color="#7C3AED", ax=axes[2], edgecolor="white")
axes[2].axvline(0, color="#E11D48", linestyle="--", linewidth=1.5)
axes[2].set_title("Residual Error Distribution (Normality Check)", fontsize=12, fontweight="bold")
axes[2].set_xlabel("Residual (mol/L)")
axes[2].set_ylabel("Density / Frequency")

plt.tight_layout()
plt.show()

<div style="padding: 12px 18px; margin: 20px 0 12px 0; color: #0F2742; font-size: 18px; font-weight: bold; background-color: #F0FDFA; border-left: 6px solid #0D9488; border-radius: 4px;">
    12.2 Production Inference Interface on top of New SMILES
</div>

In [ ]:

def predict_aqueous_solubility(smiles: str, model_path: Path = OUTPUT_DIR / "solubility_prediction_model.joblib"):
    """
    Predict aqueous solubility (log S, mol/L) for an arbitrary SMILES input.
    Validates molecular structure, builds Morgan ECFP4 fingerprint, and predicts solubility.
    """
    if not isinstance(smiles, str) or not smiles.strip():
        raise ValueError("Input SMILES must be a non-empty string.")
    
    mol = Chem.MolFromSmiles(smiles.strip())
    if mol is None:
        raise ValueError(f"Invalid molecular structure for SMILES: '{smiles}'")
    
    canonical_smi = Chem.MolToSmiles(mol, canonical=True, isomericSmiles=True)
    
    # Generate 1024-bit Morgan Fingerprint
    generator = rdFingerprintGenerator.GetMorganGenerator(radius=FP_RADIUS, fpSize=FP_NBITS)
    fp = generator.GetFingerprintAsNumPy(mol).astype(np.float32).reshape(1, -1)
    
    # Load model and predict
    model = joblib.load(model_path)
    pred_val = float(model.predict(fp)[0])
    
    # Determine solubility tier
    if pred_val > 0.0:
        tier = "Highly Soluble"
    elif pred_val >= -2.0:
        tier = "Moderately Soluble"
    elif pred_val >= -4.0:
        tier = "Slightly Soluble"
    else:
        tier = "Poorly Soluble / Insoluble"
        
    return {
        "Query_SMILES": smiles,
        "Canonical_SMILES": canonical_smi,
        "Predicted_LogS_mol_L": round(pred_val, 4),
        "Solubility_Tier": tier
    }

# Demonstration on Benchmark Pharmaceutical Compounds
demo_compounds = [
    ("Aspirin", "CC(=O)Oc1ccccc1C(=O)O"),
    ("Paracetamol", "CC(=O)Nc1ccc(O)cc1"),
    ("Caffeine", "Cn1cnc2c1c(=O)n(c(=O)n2C)C"),
    ("Atorvastatin", "CC(C)c1c(C(=O)Nc2ccccc2)c(-c2ccccc2)c(-c2ccc(F)cc2)n1CCC(O)CC(O)CC(=O)O")
]

print("--- Production Inference Pipeline Demonstration ---")
demo_results = [
    {"Compound": name, **predict_aqueous_solubility(smi)}
    for name, smi in demo_compounds
]
display(pd.DataFrame(demo_results))